# PROJEKT CZ. 2: UCZENIE MASZYNOWE

In [46]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error, recall_score, mean_absolute_error, r2_score
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import warnings

warnings.filterwarnings('ignore')

results_data = []
wyniki =[]


WCZYTANIE I PRZYGOTOWANIE DANYCH

In [47]:


df = pd.read_csv('../../data/credit_risk_dataset.csv')

# Usuwanie anomalii
df = df[df['person_age'] < 100]
df = df[df['person_emp_length'] < 60]

# Uzupełnienie braków i kodowanie zmiennych kategorycznych
df = df.fillna(df.mean(numeric_only=True)) 
df = pd.get_dummies(df, drop_first=True)
df = df.astype(float)

# Zmienne docelowe:
#   - klasyfikacja: loan_status (0/1 - czy pożyczka spłacona)
#   - regresja:     loan_int_rate (wysokość oprocentowania)
TARGET_CLF = 'loan_status'
TARGET_REG = 'loan_int_rate'

y_clf = df[TARGET_CLF].values.ravel()
y_reg = df[TARGET_REG].values.ravel()

# Usuwamy OBIE kolumny docelowe z X (uniknięcie target leakage)
X_clf = df.drop([TARGET_CLF, TARGET_REG], axis=1).values
X_reg = df.drop([TARGET_REG, TARGET_CLF], axis=1).values

# Podział 70% train / 30% test
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.3, stratify=y_clf, random_state=42)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42)

# Skalowanie (tylko kolumny numeryczne – bez 0/1)
binary_cols = [i for i in range(X_train_clf.shape[1])
               if np.isin(X_train_clf[:, i], [0, 1]).all()]
numeric_cols = [i for i in range(X_train_clf.shape[1])
                if i not in binary_cols]

DANE

In [19]:
df.head(5)

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,...,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,loan_grade_B,loan_grade_C,loan_grade_D,loan_grade_E,loan_grade_F,loan_grade_G,cb_person_default_on_file_Y
1,21.0,9600.0,5.0,1000.0,11.14,0.0,0.10,2.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,25.0,9600.0,1.0,5500.0,12.87,1.0,0.57,3.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,23.0,65500.0,4.0,35000.0,15.23,1.0,0.53,2.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,24.0,54400.0,8.0,35000.0,14.27,1.0,0.55,4.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
5,21.0,9900.0,2.0,2500.0,7.14,1.0,0.25,2.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


KLASYFIKACJA

In [48]:
scaler_clf = StandardScaler()
X_train_clf_s = X_train_clf.copy()
X_test_clf_s = X_test_clf.copy()
X_train_clf_s[:, numeric_cols] = scaler_clf.fit_transform(X_train_clf[:, numeric_cols])
X_test_clf_s[:, numeric_cols] = scaler_clf.transform(X_test_clf[:, numeric_cols])

# ===== REGRESJA =====
binary_cols_reg = [i for i in range(X_train_reg.shape[1])
                   if np.isin(X_train_reg[:, i], [0, 1]).all()]
numeric_cols_reg = [i for i in range(X_train_reg.shape[1])
                    if i not in binary_cols_reg]

scaler_reg = StandardScaler()
X_train_reg_s = X_train_reg.copy()
X_test_reg_s = X_test_reg.copy()
X_train_reg_s[:, numeric_cols_reg] = scaler_reg.fit_transform(X_train_reg[:, numeric_cols_reg])
X_test_reg_s[:, numeric_cols_reg] = scaler_reg.transform(X_test_reg[:, numeric_cols_reg])

print("Dane przygotowane.")
print(f"  Rozmiar zbioru uczącego  (clf): {X_train_clf_s.shape}") #shape zwraca (liczba wierszy, liczba kolumn)
print(f"  Rozmiar zbioru testowego (clf): {X_test_clf_s.shape}")
print(f"  Rozmiar zbioru uczącego  (reg): {X_train_reg_s.shape}")
print(f"  Rozmiar zbioru testowego (reg): {X_test_reg_s.shape}")

Dane przygotowane.
  Rozmiar zbioru uczącego  (clf): (22175, 21)
  Rozmiar zbioru testowego (clf): (9504, 21)
  Rozmiar zbioru uczącego  (reg): (22175, 21)
  Rozmiar zbioru testowego (reg): (9504, 21)


POMOCNICZE FUNKCJE

In [49]:
def print_header(title):
    print(f"\n{'=' * 60}")
    print(f"  {title}")
    print(f"{'=' * 60}")

def print_param_header(desc):
    print(f"\n--- {desc} ---")

# PROBLEM KLASYFIKACYJNY

### KNN


KNN – PARAMETR 1: liczba sąsiadów (n_neighbors)

In [22]:
print_param_header("KNN | PARAMETR 1: liczba sąsiadów (n_neighbors)")
print(f"  {'n_neighbors':<15} {'Train Acc':>12} {'Test Acc':>12}")
print(f"  {'-'*15} {'-'*12} {'-'*12}")
for k in [3, 5, 7, 9]:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train_clf_s, y_train_clf)
    tr = accuracy_score(y_train_clf, m.predict(X_train_clf_s))
    te = accuracy_score(y_test_clf, m.predict(X_test_clf_s))  
    tr_recall = recall_score(y_train_clf, m.predict(X_train_clf_s))
    te_recall = recall_score(y_test_clf, m.predict(X_test_clf_s))
    print(f"  k = {k:<11} {tr:>12.4f} {te:>12.4f}")
    print(f"  (Recall) {'':<7} {tr_recall:>12.4f} {te_recall:>12.4f}")

    results_data.append({
        'Model': 'KNN',
        'Parametr': f'n_neighbors={k}',
        'Accuracy': round(te, 4),
        'Recall_klasa_1': round(te_recall, 4)
    })


--- KNN | PARAMETR 1: liczba sąsiadów (n_neighbors) ---
  n_neighbors        Train Acc     Test Acc
  --------------- ------------ ------------
  k = 3                 0.9293       0.8790
  (Recall)               0.7270       0.5908
  k = 5                 0.9130       0.8847
  (Recall)               0.6523       0.5737
  k = 7                 0.9068       0.8874
  (Recall)               0.6178       0.5674
  k = 9                 0.9017       0.8875
  (Recall)               0.5958       0.5547


KNN – PARAMETR 2: metryka odległości

In [23]:
print_param_header("KNN | PARAMETR 2: metryka odległości (metric)")
print(f"  {'metric':<15} {'Train Acc':>12} {'Test Acc':>12}")
print(f"  {'-'*15} {'-'*12} {'-'*12}")
for metr in ['euclidean', 'manhattan', 'chebyshev', 'minkowski']:
    m = KNeighborsClassifier(n_neighbors=5, metric=metr)
    m.fit(X_train_clf_s, y_train_clf)
    tr = accuracy_score(y_train_clf, m.predict(X_train_clf_s))
    te = accuracy_score(y_test_clf, m.predict(X_test_clf_s))
    tr_recall = recall_score(y_train_clf, m.predict(X_train_clf_s))
    te_recall = recall_score(y_test_clf, m.predict(X_test_clf_s))
    print(f"  {metr:<15} {tr:>12.4f} {te:>12.4f}")
    print(f"  (Recall) {'':<7} {tr_recall:>12.4f} {te_recall:>12.4f}")
    results_data.append({
        'Model': 'KNN',
        'Parametr': f'metric={metr}',
        'Accuracy': round(te, 4),
        'Recall_klasa_1': round(te_recall, 4)
    })



--- KNN | PARAMETR 2: metryka odległości (metric) ---
  metric             Train Acc     Test Acc
  --------------- ------------ ------------
  euclidean             0.9130       0.8847
  (Recall)               0.6523       0.5737
  manhattan             0.9154       0.8884
  (Recall)               0.6575       0.5791
  chebyshev             0.9049       0.8746
  (Recall)               0.6192       0.5386
  minkowski             0.9130       0.8847
  (Recall)               0.6523       0.5737


### SVM


SVM – PARAMETR 1: parametr regularyzacji C

In [24]:
print_param_header("SVM | PARAMETR 1: parametr regularyzacji (C)")
print(f"  {'C':<15} {'Train Acc':>12} {'Test Acc':>12}")
print(f"  {'-'*15} {'-'*12} {'-'*12}")
for c in [0.1, 1.0, 10.0, 100.0]:
    m = SVC(C=c, kernel='rbf', random_state=42)
    m.fit(X_train_clf_s, y_train_clf)
    tr = accuracy_score(y_train_clf, m.predict(X_train_clf_s))
    te = accuracy_score(y_test_clf, m.predict(X_test_clf_s))
    tr_recall = recall_score(y_train_clf, m.predict(X_train_clf_s))
    te_recall = recall_score(y_test_clf, m.predict(X_test_clf_s))
    print(f"  C = {c:<11} {tr:>12.4f} {te:>12.4f}")
    print(f"  (Recall) {'':<7} {tr_recall:>12.4f} {te_recall:>12.4f}")
    results_data.append({
        'Model': 'SVM',
        'Parametr': f'C={c}',
        'Accuracy': round(te, 4),
        'Recall_klasa_1': round(te_recall, 4)
    })



--- SVM | PARAMETR 1: parametr regularyzacji (C) ---
  C                  Train Acc     Test Acc
  --------------- ------------ ------------
  C = 0.1               0.8891       0.8924
  (Recall)               0.5365       0.5439
  C = 1.0               0.9193       0.9166
  (Recall)               0.6634       0.6577
  C = 10.0              0.9344       0.9188
  (Recall)               0.7170       0.6743
  C = 100.0             0.9493       0.9129
  (Recall)               0.7766       0.6875


SVR – PARAMETR 2: rodzaj jądra (kernel)

In [25]:
print_param_header("SVM | PARAMETR 2: rodzaj jądra (kernel)")
print(f"  {'kernel':<15} {'Train Acc':>12} {'Test Acc':>12}")
print(f"  {'-'*15} {'-'*12} {'-'*12}")
for kern in ['linear', 'poly', 'rbf', 'sigmoid']:
    m = SVC(C=1.0, kernel=kern, random_state=42)
    m.fit(X_train_clf_s, y_train_clf)
    tr = accuracy_score(y_train_clf, m.predict(X_train_clf_s))
    te = accuracy_score(y_test_clf, m.predict(X_test_clf_s))
    tr_recall = recall_score(y_train_clf, m.predict(X_train_clf_s))
    te_recall = recall_score(y_test_clf, m.predict(X_test_clf_s))
    print(f"  {kern:<15} {tr:>12.4f} {te:>12.4f}")
    print(f"  (Recall) {'':<7} {tr_recall:>12.4f} {te_recall:>12.4f}")
    results_data.append({
        'Model': 'SVM',
        'Parametr': f'kernel={kern}',
        'Accuracy': round(te, 4),
        'Recall_klasa_1': round(te_recall, 4)
    })





--- SVM | PARAMETR 2: rodzaj jądra (kernel) ---
  kernel             Train Acc     Test Acc
  --------------- ------------ ------------
  linear                0.8675       0.8702
  (Recall)               0.5562       0.5605
  poly                  0.9109       0.9068
  (Recall)               0.6364       0.6284
  rbf                   0.9193       0.9166
  (Recall)               0.6634       0.6577
  sigmoid               0.7313       0.7319
  (Recall)               0.3720       0.3872


### DRZEWO DECYZYJNE

DRZEWO – PARAMETR 1: maksymalna głębokość (max_depth)

In [26]:
print_param_header("Drzewo Decyzyjne | PARAMETR 1: maksymalna głębokość (max_depth)")
print(f"  {'max_depth':<15} {'Train Acc':>12} {'Test Acc':>12}")
print(f"  {'-'*15} {'-'*12} {'-'*12}")
for depth in [3, 5, 10, None]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42)
    m.fit(X_train_clf_s, y_train_clf)
    tr = accuracy_score(y_train_clf, m.predict(X_train_clf_s))
    te = accuracy_score(y_test_clf, m.predict(X_test_clf_s))
    tr_recall = recall_score(y_train_clf, m.predict(X_train_clf_s))
    te_recall = recall_score(y_test_clf, m.predict(X_test_clf_s))
    label = str(depth) if depth is not None else 'None (brak)'
    print(f"  {label:<15} {tr:>12.4f} {te:>12.4f}")
    print(f"  (Recall) {'':<7} {tr_recall:>12.4f} {te_recall:>12.4f}")
    results_data.append({
        'Model': 'Decision Tree',
        'Parametr': f'max_depth={depth}',
        'Accuracy': round(te, 4),
        'Recall_klasa_1': round(te_recall, 4)
    })



--- Drzewo Decyzyjne | PARAMETR 1: maksymalna głębokość (max_depth) ---
  max_depth          Train Acc     Test Acc
  --------------- ------------ ------------
  3                     0.8835       0.8835
  (Recall)               0.5133       0.5264
  5                     0.9085       0.9095
  (Recall)               0.6144       0.6196
  10                    0.9379       0.9323
  (Recall)               0.7168       0.7065
  None (brak)           1.0000       0.8939
  (Recall)               1.0000       0.7642


DRZEWO – PARAMETR 2: min. liczba obserwacji w liściu (min_samples_leaf)

In [27]:
print_param_header("Drzewo Decyzyjne | PARAMETR 2: min. liczba obserwacji w liściu (min_samples_leaf)")
print(f"  {'min_samples_leaf':<20} {'Train Acc':>12} {'Test Acc':>12}")
print(f"  {'-'*20} {'-'*12} {'-'*12}")
for msl in [1, 5, 20, 50]:
    m = DecisionTreeClassifier(min_samples_leaf=msl, random_state=42)
    m.fit(X_train_clf_s, y_train_clf)
    tr = accuracy_score(y_train_clf, m.predict(X_train_clf_s))
    te = accuracy_score(y_test_clf, m.predict(X_test_clf_s))
    tr_recall = recall_score(y_train_clf, m.predict(X_train_clf_s))
    te_recall = recall_score(y_test_clf, m.predict(X_test_clf_s))
    print(f"  {msl:<20} {tr:>12.4f} {te:>12.4f}")
    print(f"  (Recall) {'':<7} {tr_recall:>12.4f} {te_recall:>12.4f}")
    results_data.append({
        'Model': 'Decision Tree',
        'Parametr': f'min_samples_leaf={msl}',
        'Accuracy': round(te, 4),
        'Recall_klasa_1': round(te_recall, 4)
    })



--- Drzewo Decyzyjne | PARAMETR 2: min. liczba obserwacji w liściu (min_samples_leaf) ---
  min_samples_leaf        Train Acc     Test Acc
  -------------------- ------------ ------------
  1                          1.0000       0.8939
  (Recall)               1.0000       0.7642
  5                          0.9509       0.9161
  (Recall)               0.8260       0.7427
  20                         0.9299       0.9259
  (Recall)               0.7182       0.7051
  50                         0.9208       0.9220
  (Recall)               0.6718       0.6763


### LAS LOSOWY

LAS LOSOWY – PARAMETR 1: liczba drzew (n_estimators)

In [28]:
print_param_header("Las Losowy | PARAMETR 1: liczba drzew (n_estimators)")
print(f"  {'n_estimators':<15} {'Train Acc':>12} {'Test Acc':>12}")
print(f"  {'-'*15} {'-'*12} {'-'*12}")
for n_est in [10, 50, 100, 200]:
    m = RandomForestClassifier(n_estimators=n_est, random_state=42, n_jobs=-1)
    m.fit(X_train_clf_s, y_train_clf)
    tr = accuracy_score(y_train_clf, m.predict(X_train_clf_s))
    te = accuracy_score(y_test_clf, m.predict(X_test_clf_s))
    tr_recall = recall_score(y_train_clf, m.predict(X_train_clf_s))
    te_recall = recall_score(y_test_clf, m.predict(X_test_clf_s))
    print(f"  {n_est:<15} {tr:>12.4f} {te:>12.4f}")
    print(f"  (Recall) {'':<7} {tr_recall:>12.4f} {te_recall:>12.4f}")
    results_data.append({
        'Model': 'Random Forest',
        'Parametr': f'n_estimators={n_est}',
        'Accuracy': round(te, 4),
        'Recall_klasa_1': round(te_recall, 4)
    })



--- Las Losowy | PARAMETR 1: liczba drzew (n_estimators) ---
  n_estimators       Train Acc     Test Acc
  --------------- ------------ ------------
  10                    0.9891       0.9286
  (Recall)               0.9504       0.7061
  50                    0.9994       0.9335
  (Recall)               0.9971       0.7222
  100                   1.0000       0.9329
  (Recall)               0.9998       0.7231
  200                   1.0000       0.9337
  (Recall)               1.0000       0.7241


LAS LOSOWY – PARAMETR 2: maksymalna głębokość (max_depth)

In [29]:
print_param_header("Las Losowy | PARAMETR 2: maksymalna głębokość drzew (max_depth)")
print(f"  {'max_depth':<15} {'Train Acc':>12} {'Test Acc':>12}")
print(f"  {'-'*15} {'-'*12} {'-'*12}")
for depth in [3, 5, 10, None]:
    m = RandomForestClassifier(n_estimators=100, max_depth=depth,
                               random_state=42, n_jobs=-1)
    m.fit(X_train_clf_s, y_train_clf)
    tr = accuracy_score(y_train_clf, m.predict(X_train_clf_s))
    te = accuracy_score(y_test_clf, m.predict(X_test_clf_s))
    tr_recall = recall_score(y_train_clf, m.predict(X_train_clf_s))
    te_recall = recall_score(y_test_clf, m.predict(X_test_clf_s))
    label = str(depth) if depth is not None else 'None (brak)'
    print(f"  {label:<15} {tr:>12.4f} {te:>12.4f}")
    print(f"  (Recall) {'':<7} {tr_recall:>12.4f} {te_recall:>12.4f}")
    results_data.append({
        'Model': 'Random Forest',
        'Parametr': f'max_depth={depth}',
        'Accuracy': round(te, 4),
        'Recall_klasa_1': round(te_recall, 4)
    })




--- Las Losowy | PARAMETR 2: maksymalna głębokość drzew (max_depth) ---
  max_depth          Train Acc     Test Acc
  --------------- ------------ ------------
  3                     0.8598       0.8583
  (Recall)               0.3498       0.3428
  5                     0.8924       0.8927
  (Recall)               0.5081       0.5083
  10                    0.9351       0.9276
  (Recall)               0.7065       0.6812
  None (brak)           1.0000       0.9329
  (Recall)               0.9998       0.7231


# PROBLEM REGRESYJNY

### KNN

PARAMETR 1: liczba sąsiadów (n_neighbors)

In [50]:
print_param_header("KNN | PARAMETR 1: liczba sąsiadów (n_neighbors)")
print(f"  {'n_neighbors':<12} {'RMSE_train':>12} {'RMSE_test':>12} {'MAE_train':>12} {'MAE_test':>12} {'R2_train':>12} {'R2_test':>12}")
print(f"  {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")
for k in [3, 5, 7, 9]:
    m = KNeighborsRegressor(n_neighbors=k)
    m.fit(X_train_reg_s, y_train_reg)
    #comment
    y_tr = m.predict(X_train_reg_s)
    y_te = m.predict(X_test_reg_s)
    
    rmse_tr = np.sqrt(mean_squared_error(y_train_reg, y_tr))
    rmse_te = np.sqrt(mean_squared_error(y_test_reg, y_te))
    mae_tr = mean_absolute_error(y_train_reg, y_tr)
    mae_te = mean_absolute_error(y_test_reg, y_te)
    r2_tr = r2_score(y_train_reg, y_tr)
    r2_te = r2_score(y_test_reg, y_te)
    
    print(f"  k = {k:<8} {rmse_tr:>12.4f} {rmse_te:>12.4f} {mae_tr:>12.4f} {mae_te:>12.4f} {r2_tr:>12.4f} {r2_te:>12.4f}")

    wyniki.append({
        'model': 'KNN',
        'parametr': 'n_neighbors',
        'wartosc': k,
        'RMSE_train': rmse_tr,
        'RMSE_test': rmse_te,
        'MAE_train': mae_tr,
        'MAE_test': mae_te,
        'R2_train': r2_tr,
        'R2_test': r2_te
    })


--- KNN | PARAMETR 1: liczba sąsiadów (n_neighbors) ---
  n_neighbors    RMSE_train    RMSE_test    MAE_train     MAE_test     R2_train      R2_test
  ------------ ------------ ------------ ------------ ------------ ------------ ------------
  k = 3              1.3390       1.9656       0.9201       1.3405       0.8103       0.5875
  k = 5              1.5416       1.9445       1.0542       1.3216       0.7485       0.5964
  k = 7              1.6533       1.9376       1.1243       1.3149       0.7108       0.5992
  k = 9              1.7261       1.9530       1.1717       1.3248       0.6847       0.5928


PARAMETR 2: metryka odległości (metric)

In [51]:
print_param_header("KNN | PARAMETR 2: metryka odległości (metric)")
print(f"  {'metric':<12} {'RMSE_train':>12} {'RMSE_test':>12} {'MAE_train':>12} {'MAE_test':>12} {'R2_train':>12} {'R2_test':>12}")
print(f"  {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")
for metr in ['euclidean', 'manhattan', 'chebyshev', 'minkowski']:
    m = KNeighborsRegressor(n_neighbors=5, metric=metr)
    m.fit(X_train_reg_s, y_train_reg)
    
    y_tr = m.predict(X_train_reg_s)
    y_te = m.predict(X_test_reg_s)
    
    rmse_tr = np.sqrt(mean_squared_error(y_train_reg, y_tr))
    rmse_te = np.sqrt(mean_squared_error(y_test_reg, y_te))
    mae_tr = mean_absolute_error(y_train_reg, y_tr)
    mae_te = mean_absolute_error(y_test_reg, y_te)
    r2_tr = r2_score(y_train_reg, y_tr)
    r2_te = r2_score(y_test_reg, y_te)
    
    print(f"  {metr:<12} {rmse_tr:>12.4f} {rmse_te:>12.4f} {mae_tr:>12.4f} {mae_te:>12.4f} {r2_tr:>12.4f} {r2_te:>12.4f}")

    wyniki.append({
        'model': 'KNN',
        'parametr': 'metric',
        'wartosc': metr,
        'RMSE_train': rmse_tr,
        'RMSE_test': rmse_te,
        'MAE_train': mae_tr,
        'MAE_test': mae_te,
        'R2_train': r2_tr,
        'R2_test': r2_te
    })


--- KNN | PARAMETR 2: metryka odległości (metric) ---
  metric         RMSE_train    RMSE_test    MAE_train     MAE_test     R2_train      R2_test
  ------------ ------------ ------------ ------------ ------------ ------------ ------------
  euclidean          1.5416       1.9445       1.0542       1.3216       0.7485       0.5964
  manhattan          1.5413       1.9325       1.0603       1.3238       0.7486       0.6013
  chebyshev          1.5563       1.9451       1.0744       1.3502       0.7437       0.5961
  minkowski          1.5416       1.9445       1.0542       1.3216       0.7485       0.5964


### SVR

PARAMETR 1: parametr regularyzacji C

In [55]:
print_param_header("SVR | PARAMETR 1: parametr regularyzacji (C)")
print(f"  {'C':<12} {'RMSE_train':>12} {'RMSE_test':>12} {'MAE_train':>12} {'MAE_test':>12} {'R2_train':>12} {'R2_test':>12}")
print(f"  {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")
for c in [0.1, 1.0, 10.0, 100.0]:
    m = SVR(C=c, kernel='rbf')
    m.fit(X_train_reg_s, y_train_reg)
    
    y_tr = m.predict(X_train_reg_s)
    y_te = m.predict(X_test_reg_s)
    
    rmse_tr = np.sqrt(mean_squared_error(y_train_reg, y_tr))
    rmse_te = np.sqrt(mean_squared_error(y_test_reg, y_te))
    mae_tr = mean_absolute_error(y_train_reg, y_tr)
    mae_te = mean_absolute_error(y_test_reg, y_te)
    r2_tr = r2_score(y_train_reg, y_tr)
    r2_te = r2_score(y_test_reg, y_te)
    
    print(f"  {c:<12} {rmse_tr:>12.4f} {rmse_te:>12.4f} {mae_tr:>12.4f} {mae_te:>12.4f} {r2_tr:>12.4f} {r2_te:>12.4f}")

    wyniki.append({
        'model': 'SVR',
        'parametr': 'C',
        'wartosc': c,
        'RMSE_train': rmse_tr,
        'RMSE_test': rmse_te,
        'MAE_train': mae_tr,
        'MAE_test': mae_te,
        'R2_train': r2_tr,
        'R2_test': r2_te
    })


--- SVR | PARAMETR 1: parametr regularyzacji (C) ---
  C              RMSE_train    RMSE_test    MAE_train     MAE_test     R2_train      R2_test
  ------------ ------------ ------------ ------------ ------------ ------------ ------------
  0.1                1.7447       1.7246       1.2084       1.2102       0.6779       0.6825
  1.0                1.3851       1.4294       0.9631       1.0134       0.7970       0.7819
  10.0               1.2183       1.3706       0.8327       0.9845       0.8429       0.7995
  100.0              1.1166       1.4486       0.7268       1.0464       0.8681       0.7760


PARAMETR 2: rodzaj jądra (kernel)

In [56]:
print_param_header("SVR | PARAMETR 2: rodzaj jądra (kernel)")
print(f"  {'kernel':<12} {'RMSE_train':>12} {'RMSE_test':>12} {'MAE_train':>12} {'MAE_test':>12} {'R2_train':>12} {'R2_test':>12}")
print(f"  {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")
for kern in ['linear', 'poly', 'rbf']:
    m = SVR(C=1.0, kernel=kern)
    m.fit(X_train_reg_s, y_train_reg)
    
    y_tr = m.predict(X_train_reg_s)
    y_te = m.predict(X_test_reg_s)
    ###
    rmse_tr = np.sqrt(mean_squared_error(y_train_reg, y_tr))
    rmse_te = np.sqrt(mean_squared_error(y_test_reg, y_te))
    mae_tr = mean_absolute_error(y_train_reg, y_tr)
    mae_te = mean_absolute_error(y_test_reg, y_te)
    r2_tr = r2_score(y_train_reg, y_tr)
    r2_te = r2_score(y_test_reg, y_te)
    
    print(f"  {kern:<12} {rmse_tr:>12.4f} {rmse_te:>12.4f} {mae_tr:>12.4f} {mae_te:>12.4f} {r2_tr:>12.4f} {r2_te:>12.4f}")

    wyniki.append({
        'model': 'SVR',
        'parametr': 'kernel',
        'wartosc': kern,
        'RMSE_train': rmse_tr,
        'RMSE_test': rmse_te,
        'MAE_train': mae_tr,
        'MAE_test': mae_te,
        'R2_train': r2_tr,
        'R2_test': r2_te
    })


--- SVR | PARAMETR 2: rodzaj jądra (kernel) ---
  kernel         RMSE_train    RMSE_test    MAE_train     MAE_test     R2_train      R2_test
  ------------ ------------ ------------ ------------ ------------ ------------ ------------
  linear             1.3063       1.3183       0.9281       0.9360       0.8194       0.8145
  poly               1.6077       1.6961       1.1479       1.2081       0.7265       0.6929
  rbf                1.3851       1.4294       0.9631       1.0134       0.7970       0.7819


### DRZEWO DECYZYJNE

PARAMETR 1: maksymalna głębokość (max_depth)

In [57]:
print_param_header("Drzewo Decyzyjne | PARAMETR 1: maksymalna głębokość (max_depth)")
print(f"  {'max_depth':<12} {'RMSE_train':>12} {'RMSE_test':>12} {'MAE_train':>12} {'MAE_test':>12} {'R2_train':>12} {'R2_test':>12}")
print(f"  {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")
for depth in [3, 5, 10, None]:
    m = DecisionTreeRegressor(max_depth=depth, random_state=42)
    m.fit(X_train_reg_s, y_train_reg)
    
    y_tr = m.predict(X_train_reg_s)
    y_te = m.predict(X_test_reg_s)
    
    rmse_tr = np.sqrt(mean_squared_error(y_train_reg, y_tr))
    rmse_te = np.sqrt(mean_squared_error(y_test_reg, y_te))
    mae_tr = mean_absolute_error(y_train_reg, y_tr)
    mae_te = mean_absolute_error(y_test_reg, y_te)
    r2_tr = r2_score(y_train_reg, y_tr)
    r2_te = r2_score(y_test_reg, y_te)
    
    label = str(depth) if depth is not None else 'None'
    print(f"  {label:<12} {rmse_tr:>12.4f} {rmse_te:>12.4f} {mae_tr:>12.4f} {mae_te:>12.4f} {r2_tr:>12.4f} {r2_te:>12.4f}")

    wyniki.append({
        'model': 'Decision Tree',
        'parametr': 'max_depth',
        'wartosc': str(depth),
        'RMSE_train': rmse_tr,
        'RMSE_test': rmse_te,
        'MAE_train': mae_tr,
        'MAE_test': mae_te,
        'R2_train': r2_tr,
        'R2_test': r2_te
    })


--- Drzewo Decyzyjne | PARAMETR 1: maksymalna głębokość (max_depth) ---
  max_depth      RMSE_train    RMSE_test    MAE_train     MAE_test     R2_train      R2_test
  ------------ ------------ ------------ ------------ ------------ ------------ ------------
  3                  2.1558       2.1475       1.6677       1.6601       0.5082       0.5077
  5                  1.4943       1.4718       0.9900       0.9916       0.7637       0.7687
  10                 1.2155       1.3612       0.8771       0.9721       0.8437       0.8022
  None               0.0199       1.8450       0.0003       1.3413       1.0000       0.6366


PARAMETR 2: min. liczba obserwacji w liściu (min_samples_leaf)

In [58]:
print_param_header("Drzewo Decyzyjne | PARAMETR 2: min. liczba obserwacji w liściu (min_samples_leaf)")
print(f"  {'min_samp_leaf':<12} {'RMSE_train':>12} {'RMSE_test':>12} {'MAE_train':>12} {'MAE_test':>12} {'R2_train':>12} {'R2_test':>12}")
print(f"  {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")
for msl in [1, 5, 20, 50]:
    m = DecisionTreeRegressor(min_samples_leaf=msl, random_state=42)
    m.fit(X_train_reg_s, y_train_reg)
    
    y_tr = m.predict(X_train_reg_s)
    y_te = m.predict(X_test_reg_s)
    
    rmse_tr = np.sqrt(mean_squared_error(y_train_reg, y_tr))
    rmse_te = np.sqrt(mean_squared_error(y_test_reg, y_te))
    mae_tr = mean_absolute_error(y_train_reg, y_tr)
    mae_te = mean_absolute_error(y_test_reg, y_te)
    r2_tr = r2_score(y_train_reg, y_tr)
    r2_te = r2_score(y_test_reg, y_te)
    
    print(f"  {msl:<12} {rmse_tr:>12.4f} {rmse_te:>12.4f} {mae_tr:>12.4f} {mae_te:>12.4f} {r2_tr:>12.4f} {r2_te:>12.4f}")

    wyniki.append({
        'model': 'Decision Tree',
        'parametr': 'min_samples_leaf',
        'wartosc': msl,
        'RMSE_train': rmse_tr,
        'RMSE_test': rmse_te,
        'MAE_train': mae_tr,
        'MAE_test': mae_te,
        'R2_train': r2_tr,
        'R2_test': r2_te
    })


--- Drzewo Decyzyjne | PARAMETR 2: min. liczba obserwacji w liściu (min_samples_leaf) ---
  min_samp_leaf   RMSE_train    RMSE_test    MAE_train     MAE_test     R2_train      R2_test
  ------------ ------------ ------------ ------------ ------------ ------------ ------------
  1                  0.0199       1.8450       0.0003       1.3413       1.0000       0.6366
  5                  0.9463       1.5381       0.6987       1.1491       0.9052       0.7474
  20                 1.1970       1.3739       0.8874       1.0160       0.8484       0.7985
  50                 1.3010       1.3702       0.9334       0.9867       0.8209       0.7996


### LAS LOSOWY

PARAMETR 1: liczba drzew (n_estimators)

In [59]:
print_param_header("Las Losowy | PARAMETR 1: liczba drzew (n_estimators)")
print(f"  {'n_estimators':<12} {'RMSE_train':>12} {'RMSE_test':>12} {'MAE_train':>12} {'MAE_test':>12} {'R2_train':>12} {'R2_test':>12}")
print(f"  {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")
for n_est in [10, 50, 100, 200]:
    m = RandomForestRegressor(n_estimators=n_est, random_state=42, n_jobs=-1)
    m.fit(X_train_reg_s, y_train_reg)
    
    y_tr = m.predict(X_train_reg_s)
    y_te = m.predict(X_test_reg_s)
    
    rmse_tr = np.sqrt(mean_squared_error(y_train_reg, y_tr))
    rmse_te = np.sqrt(mean_squared_error(y_test_reg, y_te))
    mae_tr = mean_absolute_error(y_train_reg, y_tr)
    mae_te = mean_absolute_error(y_test_reg, y_te)
    r2_tr = r2_score(y_train_reg, y_tr)
    r2_te = r2_score(y_test_reg, y_te)
    
    print(f"  {n_est:<12} {rmse_tr:>12.4f} {rmse_te:>12.4f} {mae_tr:>12.4f} {mae_te:>12.4f} {r2_tr:>12.4f} {r2_te:>12.4f}")

    wyniki.append({
        'model': 'Random Forest',
        'parametr': 'n_estimators',
        'wartosc': n_est,
        'RMSE_train': rmse_tr,
        'RMSE_test': rmse_te,
        'MAE_train': mae_tr,
        'MAE_test': mae_te,
        'R2_train': r2_tr,
        'R2_test': r2_te
    })


--- Las Losowy | PARAMETR 1: liczba drzew (n_estimators) ---
  n_estimators   RMSE_train    RMSE_test    MAE_train     MAE_test     R2_train      R2_test
  ------------ ------------ ------------ ------------ ------------ ------------ ------------
  10                 0.5851       1.3775       0.4030       1.0153       0.9638       0.7974
  50                 0.5022       1.3225       0.3620       0.9661       0.9733       0.8133
  100                0.4920       1.3150       0.3566       0.9598       0.9744       0.8154
  200                0.4865       1.3129       0.3532       0.9575       0.9750       0.8160


PARAMETR 2: maksymalna głębokość (max_depth)

In [60]:
print_param_header("Las Losowy | PARAMETR 2: maksymalna głębokość drzew (max_depth)")
print(f"  {'max_depth':<12} {'RMSE_train':>12} {'RMSE_test':>12} {'MAE_train':>12} {'MAE_test':>12} {'R2_train':>12} {'R2_test':>12}")
print(f"  {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")
for depth in [3, 5, 10, None]:
    m = RandomForestRegressor(n_estimators=100, max_depth=depth, random_state=42, n_jobs=-1)
    m.fit(X_train_reg_s, y_train_reg)
    
    y_tr = m.predict(X_train_reg_s)
    y_te = m.predict(X_test_reg_s)
    
    rmse_tr = np.sqrt(mean_squared_error(y_train_reg, y_tr))
    rmse_te = np.sqrt(mean_squared_error(y_test_reg, y_te))
    mae_tr = mean_absolute_error(y_train_reg, y_tr)
    mae_te = mean_absolute_error(y_test_reg, y_te)
    r2_tr = r2_score(y_train_reg, y_tr)
    r2_te = r2_score(y_test_reg, y_te)
    
    label = str(depth) if depth is not None else 'None'
    print(f"  {label:<12} {rmse_tr:>12.4f} {rmse_te:>12.4f} {mae_tr:>12.4f} {mae_te:>12.4f} {r2_tr:>12.4f} {r2_te:>12.4f}")

    wyniki.append({
        'model': 'Random Forest',
        'parametr': 'max_depth',
        'wartosc': str(depth),
        'RMSE_train': rmse_tr,
        'RMSE_test': rmse_te,
        'MAE_train': mae_tr,
        'MAE_test': mae_te,
        'R2_train': r2_tr,
        'R2_test': r2_te
    })


--- Las Losowy | PARAMETR 2: maksymalna głębokość drzew (max_depth) ---
  max_depth      RMSE_train    RMSE_test    MAE_train     MAE_test     R2_train      R2_test
  ------------ ------------ ------------ ------------ ------------ ------------ ------------
  3                  2.1556       2.1472       1.6673       1.6595       0.5083       0.5078
  5                  1.4906       1.4645       0.9872       0.9876       0.7649       0.7710
  10                 1.1960       1.3062       0.8758       0.9430       0.8486       0.8179
  None               0.4920       1.3150       0.3566       0.9598       0.9744       0.8154


In [61]:
df_results = pd.DataFrame(results_data)
df_wyniki = pd.DataFrame(wyniki)

df_results.to_csv('porownanie_modeli.csv', index=False, sep=';', decimal=',')
df_wyniki.to_csv('wyniki_regresja.csv', index=False, sep=';', decimal=',')